In [3]:
import pandas as pd
df = pd.read_csv('../../02_Data/processed/real_final_ml.csv')

In [7]:
for col in df.columns:
    print(col)

enableBoardGameProperties
minPlayers
maxPlayers
minAge
playTime
fundedInSeconds
isDiscounted
previous_campaigns_count
duration_days
softclose
campaignGoal_usd_1m
fundsGathered_usd_1m
price_usd_1m
campaignGoal_usd_6m
fundsGathered_usd_6m
price_usd_6m
creator_id_0
creator_id_1
is_pledge_master_0
is_pledge_master_1
is_backer_0
is_backer_1
is_prior_backer_0
is_prior_backer_1
is_pathfinder_0
is_pathfinder_1
likes_0
likes_1
Product_Question_count_0
Product_Question_count_1
Suggestion_Idea_count_0
Suggestion_Idea_count_1
Praise_Support_count_0
Praise_Support_count_1
Shipping_Fulfillment_count_0
Shipping_Fulfillment_count_1
Complaint_Refund_count_0
Complaint_Refund_count_1
Spam_Irrelevant_count_0
Spam_Irrelevant_count_1
긍정_0
긍정_1
부정_0
부정_1
중립_0
중립_1
Campaign / Pledge_0
Campaign / Pledge_1
Community Reaction_0
Community Reaction_1
Components & Production_0
Components & Production_1
Gameplay / Rules / Content_0
Gameplay / Rules / Content_1
Language & Localization_0
Language & Localization_1
Sche

In [10]:
# 1달 지표 y=fundsGathered_usd_1m or fundsGathered_usd_1m/price_usd_1m
cols_1m = ['campaignGoal_usd_1m', 'fundsGathered_usd_1m', 'price_usd_1m']
# 6달 지표
cols_6m = ['campaignGoal_usd_6m', 'fundsGathered_usd_6m', 'price_usd_6m']

cols_currency = ['currencySymbol_AUD','currencySymbol_CAD','currencySymbol_EUR','currencySymbol_GBP','currencySymbol_PLN','currencySymbol_USD']
cols_tags = ['4XAR_Next', 'ARNext26', 'Action', 'Adventure', 'Area_Control', 'Asymmetric', 
             'Campaign', 'Card_Game', 'Cats', 'Civilization', 'Collectible', 'Collectible_Models', 
             'Competitive', 'Cooperative', 'Deck_Building', 'Deduction', 'Dexterity', 'Dice_Game', 
             'Digital', 'Economic', 'Educational', 'Exploration', 'Family', 'Fantasy', 'Feast', 
             'First-Timers', 'Game_Components', 'History', 'Horror', 'Humor', 'Legacy', 'Logical', 
             'MOBA', 'Major_Creators', 'Media', 'Modern', 'Multiplayer', 'Mythology', 'Narrative', 
             'OctoberFeast25', 'Paint', 'Party_game', 'Political', 'Print_and_Play', 'RPG', 'Racing', 
             'Reprints', 'Resource_management', 'Science_Fiction', 'Set_Collection', 'Sport', 
             'Strategy', 'Survival', 'TTRPG', 'Terrain_Building', 'Tower_Defense', 'Video_Game', 
             'Video_Game_Theme', 'Wargame', 'WinterFeast2026', 'Worker_placement', 'dinotuesday']

cols_base = [col for col in df.columns if col not in (cols_1m + cols_6m + cols_tags)]

In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
# 🌟 LightGBM 대신 호환성 문제가 절대 없는 사이킷런의 고성능 부스팅 모델만 사용합니다.
from sklearn.ensemble import HistGradientBoostingRegressor 

experimental_results = [] 

def run_ml_experiment(df, experiment_name, target_period='1m', include_tags=True, include_currency=True):
    """
    호환성 버그가 있는 LightGBM을 완전히 제거하고, 
    동일한 원리로 동작하는 HistGradientBoostingRegressor로 고정된 안전한 함수입니다.
    """
    # [A. 타겟 설정 및 기본 피처 구성]
    if target_period == '1m':
        y = df['fundsGathered_usd_1m']
        X_cols = cols_base.copy()
        X_cols.extend(['campaignGoal_usd_1m', 'price_usd_1m']) 
    else:
        y = df['fundsGathered_usd_6m']
        X_cols = cols_base.copy()
        X_cols.extend(['campaignGoal_usd_6m', 'price_usd_6m'])
        
    # [B. 옵션에 따라 피처 그룹 추가/제외]
    if include_tags:
        X_cols.extend(cols_tags)
    if include_currency:
        X_cols.extend(cols_currency)
        
    # 중복 피처 제거 및 존재하는 컬럼만 필터링 (안전장치)
    X_cols = list(dict.fromkeys(X_cols))
    X_cols = [col for col in X_cols if col in df.columns]
    
    X = df[X_cols]
    
    # [C. 데이터 분할]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # [D. 모델 선언 및 학습]
    # 에러를 유발하는 LightGBM을 쓰지 않고 안전한 모델로 학습합니다.
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    # [E. 예측 및 성능 평가]
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    # [F. 결과를 딕셔너리에 기록]
    result = {
        '실험명': experiment_name,
        '예측기간': target_period,
        '사용한 피처 개수': X.shape[1],
        '태그포함여부': include_tags,
        '화폐포함여부': include_currency,
        'RMSE': round(rmse, 2),
        'R2_Score': round(r2, 4)
    }
    experimental_results.append(result)
    
    print(f"✅ [{experiment_name}] 실험 완료! -> R²: {round(r2, 4)}, 피처수: {X.shape[1]}")
    return model

In [17]:

def run_ml_experiment(df, experiment_name, target_period='1m', include_tags=True, include_currency=True):
    """
    experiment_name: 이번 실험의 이름 (예: "1달기준_태그포함_기본모델")
    target_period: '1m' (1달 타겟) 또는 '6m' (6달 타겟)
    include_tags: 태그 피처(63개)를 포함할지 여부 (True/False)
    include_currency: 화폐 피처를 포함할지 여부 (True/False)
    """
    
    # A. 타겟 설정 및 기본 피처 구성
    if target_period == '1m':
        y = df['fundsGathered_usd_1m']  # 예측할 y값
        X_cols = cols_base.copy()
        # 1m 예측 시 목표액(Goal)과 가격(Price)은 힌트로 줄 수 있다면 추가
        X_cols.extend(['campaignGoal_usd_1m', 'price_usd_1m']) 
    else:
        y = df['fundsGathered_usd_6m']  # 예측할 y값
        X_cols = cols_base.copy()
        X_cols.extend(['campaignGoal_usd_6m', 'price_usd_6m'])
        
    # B. 옵션에 따라 피처 그룹 추가/제외
    if include_tags:
        X_cols.extend(cols_tags)
    if include_currency:
        X_cols.extend(cols_currency)
        
    X = df[X_cols]
    
    # C. 데이터 분할 (Train / Test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # D. 모델 선언 및 학습
    model = LGBMRegressor(random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    
    # E. 예측 및 성능 평가
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    # F. 결과를 딕셔너리에 기록
    result = {
        '실험명': experiment_name,
        '예측기간': target_period,
        '사용한 피처 개수': X.shape[1],
        '태그포함여부': include_tags,
        '화폐포함여부': include_currency,
        'RMSE': round(rmse, 2),
        'R2_Score': round(r2, 4)
    }
    experimental_results.append(result)
    
    print(f"✅ [{experiment_name}] 실험 완료! -> R²: {round(r2, 4)}, 피처수: {X.shape[1]}")
    return model

In [21]:
# 실험 1: 1달 기준 전체 변수 다 넣고 돌리기
run_ml_experiment(df, "1달_전체피처", target_period='1m', include_tags=True, include_currency=True)

# 실험 2: 1달 기준 태그의 영향력 확인 (태그 빼고 돌리기)
run_ml_experiment(df, "1달_태그제외", target_period='1m', include_tags=False, include_currency=True)

✅ [1달_전체피처] 실험 완료! -> R²: 0.7432, 피처수: 125
✅ [1달_태그제외] 실험 완료! -> R²: 0.7472, 피처수: 81


HistGradientBoostingRegressor(random_state=42)